# 4. Ensembles de Arboles de Decision

Un arbol de decisión es un modelo débil, el aumento del poder predictivo proviene al ensamblar varios arboles de decisión.
<br> Si promedio n arboles identicos, el resultados es exactamente el mismo que utilizar un solo arbol, necesito PERTURBAR cada arbol para disponer de variablidad

la variabilidad provendrá de estas fuentes:


*   Perturbar el dataset
*   Perturbar el algoritmo del arbol
*   Perturbar el dataset y el algoritmo del arbol al mismo tiempo

Se verán estos tres algoritmos


*   Arboles Azarosos
*   Random Forest
*   Gradient Boosting of Decision Trees

#### 4.01 Seteo del ambiente en Google Colab

Esta parte se debe correr con el runtime en Python3
<br>Ir al menu, Runtime -> Change Runtime Type -> Runtime type ->  **Python 3**

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [ ]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Para correr la siguiente celda es fundamental en Arranque en Frio haber copiado el archivo kaggle.json al Google Drive, en la carpeta indicada en el instructivo

<br>los siguientes comando estan en shell script de Linux
*   Crear las carpetas en el Google Drive
*   "instalar" el archivo kaggle.json desde el Google Drive a la virtual machine para que pueda ser utilizado por la libreria  kaggle de Python
*   Bajar el  **dataset_pequeno**  al  Google Drive  y tambien al disco local de la virtual machine que esta corriendo Google Colab



In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/labo1"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/labo1" /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json

mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo1/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}


# hago la descarga efectiva, llamando a descargar()
descargar  "dataset_pequeno.csv"



---



## 4.02 Arboles Azarosos

Arboles Azarosos es el nombre de un algoritmo trivial (por favor NO confundir con Random Forest)
Qué tipo de perturbaciones se realizan en Arboles Azarosos
* Se perturba el dataset
* No se perturba el algoritmo, es siempre rpart original

Cada  arbolito de  Arboles Azarosos se entrena sobre un dataset perturbado,  que tiene exactamente la misma cantidad de registros pero solo un subconjunto de los atributos (campos)  del dataset, tomados al azar, de los originales.
<br> En esta primera corrida, se construira cada arbol en un dataset utilizando el 50% de los campos

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [13]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Mon Oct 05 17:29:32 2026"

In [14]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,743175,39.7,1479647,79.1,1479647,79.1
Vcells,1395370,10.7,8388608,64.0,2556129,19.6


In [15]:
# cargo las librerias que necesito
require("data.table")
require("rpart")

## 4.02 Parámetros del Grid Search

Cada combinación se evalúa con **32 Árboles Azarosos**.

Se usa una sola `semilla_primigenia`, fijada al comienzo de cada combinación. Así todas las combinaciones utilizan exactamente la misma secuencia de 32 subconjuntos aleatorios de variables.

La partición estratificada 70/30 también se realiza una sola vez y queda fija para todo el grid search.


In [16]:
PARAM <- list()

# semilla inicial única
PARAM$semilla_primigenia <- 149911

# cantidad de árboles azarosos por combinación
PARAM$num_trees_max <- 32L

# porcentaje de training
PARAM$training_pct <- 70L

# función de ganancia de la cátedra
PARAM$prob_corte <- 0.025
PARAM$ganancia_acierto <- 975000
PARAM$costo_error <- -25000

# grilla
PARAM$grid_feature_fraction <- c(0.5, 0.7, 0.9)
PARAM$grid_maxdepth <- c(6, 7, 8, 9)
PARAM$grid_cp <- -1
PARAM$grid_minsplit <- c(200, 400, 600, 800, 1000)
PARAM$grid_ratio_bucket <- c(1/3, 0.40)


In [17]:
# carpeta exclusiva para este experimento
setwd("/content/buckets/b1/exp")
experimento <- "exp4021_gridsearch_semilla_unica"
dir.create(experimento, showWarnings = FALSE)
setwd(paste0("/content/buckets/b1/exp/", experimento))

getwd()


[1] "/home/ds/buckets/b1/exp/exp4021_gridsearch_semilla_unica"

In [18]:
# lectura del dataset
dataset <- fread("/content/datasets/dataset_pequeno.csv")

# período con clase conocida
dtrain_full <- dataset[foto_mes == 202107]

# período a predecir para Kaggle
dfuture <- dataset[foto_mes == 202109]
dfuture[, clase_ternaria := NA]

# variables candidatas
campos_buenos <- copy(
  setdiff(
    colnames(dtrain_full),
    c("clase_ternaria", "fold")
  )
)

cat("Registros 202107:", nrow(dtrain_full), "\n")
cat("Registros 202109:", nrow(dfuture), "\n")
cat("Campos candidatos:", length(campos_buenos), "\n")


Registros 202107: 164479 
Registros 202109: 165076 
Campos candidatos: 154 


### Función de partición estratificada de la cátedra


In [19]:
particionar <- function(
    data, division, agrupa = "",
    campo = "fold", start = 1, seed = NA) {

  if (!is.na(seed)) set.seed(seed)

  bloque <- unlist(
    mapply(
      function(x, y) {
        rep(y, x)
      },
      division,
      seq(
        from = start,
        length.out = length(division)
      )
    )
  )

  data[
    ,
    (campo) := sample(
      rep(
        bloque,
        ceiling(.N / length(bloque))
      )
    )[1:.N],
    by = agrupa
  ]
}


### Partición fija 70/30

La división se hace una sola vez. Todas las combinaciones del grid search se entrenan y evalúan sobre exactamente los mismos registros.


In [20]:
datos_grid <- copy(dtrain_full)

particionar(
  datos_grid,
  division = c(
    PARAM$training_pct,
    100L - PARAM$training_pct
  ),
  agrupa = "clase_ternaria",
  seed = PARAM$semilla_primigenia
)

dtrain <- datos_grid[fold == 1]
dtest <- datos_grid[fold == 2]

cat("Training:", nrow(dtrain), "\n")
cat("Testing:", nrow(dtest), "\n")

print(
  datos_grid[
    ,
    .N,
    by = .(fold, clase_ternaria)
  ][order(fold, clase_ternaria)]
)


Training: 115136 
Testing: 49343 
    fold clase_ternaria      N
   <num>         <char>  <int>
1:     1         BAJA+1    768
2:     1         BAJA+2    915
3:     1       CONTINUA 113453
4:     2         BAJA+1    330
5:     2         BAJA+2    389
6:     2       CONTINUA  48624


### Estimar la ganancia de un Árbol Azaroso

Cada árbol selecciona aleatoriamente una proporción de variables determinada por `feature_fraction`.

Dentro de esta función **no se vuelve a ejecutar `set.seed()`**. Esto permite que los 32 árboles de una combinación vayan consumiendo secuencialmente distintos sorteos de variables.


In [21]:
ArbolAzarosoEstimarGanancia <- function(
  param_rpart,
  feature_fraction
) {

  qty_campos <- max(
    1L,
    as.integer(
      length(campos_buenos) * feature_fraction
    )
  )

  campos_random <- sample(
    campos_buenos,
    qty_campos,
    replace = FALSE
  )

  formulita <- paste0(
    "clase_ternaria ~ ",
    paste(
      campos_random,
      collapse = " + "
    )
  )

  modelo <- rpart(
    formulita,
    data = dtrain,
    xval = 0,
    control = param_rpart
  )

  prediccion <- predict(
    modelo,
    dtest,
    type = "prob"
  )

  prob_baja2 <- prediccion[, "BAJA+2"]

  # ganancia sobre testing
  ganancia_test <- dtest[
    ,
    sum(
      ifelse(
        prob_baja2 > PARAM$prob_corte,
        ifelse(
          clase_ternaria == "BAJA+2",
          PARAM$ganancia_acierto,
          PARAM$costo_error
        ),
        0
      )
    )
  ]

  # normalización al tamaño completo
  ganancia_test_normalizada <- ganancia_test /
    ((100 - PARAM$training_pct) / 100)

  return(
    list(
      testing = nrow(dtest),
      testing_pos = dtest[
        clase_ternaria == "BAJA+2",
        .N
      ],
      envios = sum(
        prob_baja2 > PARAM$prob_corte
      ),
      aciertos = sum(
        prob_baja2 > PARAM$prob_corte &
          dtest$clase_ternaria == "BAJA+2"
      ),
      ganancia = ganancia_test_normalizada
    )
  )
}


### Evaluar una combinación con 32 árboles

La misma semilla se reinicia **una sola vez al comenzar cada combinación**.

Por eso:

- árbol 1 de cada combinación usa el mismo primer sorteo de variables;
- árbol 2 usa el mismo segundo sorteo;
- ...
- árbol 32 usa el mismo trigésimo segundo sorteo.

Esto hace que la comparación entre configuraciones sea justa y reproducible.


In [10]:
EvaluarCombinacion32 <- function(
  iteracion_actual,
  param_rpart,
  feature_fraction
) {

  detalle <- data.table()

  # una única semilla inicial por combinación
  set.seed(PARAM$semilla_primigenia)

  for (
    iarbol in seq_len(
      PARAM$num_trees_max
    )
  ) {

    resultado <- ArbolAzarosoEstimarGanancia(
      param_rpart = param_rpart,
      feature_fraction = feature_fraction
    )

    detalle <- rbind(
      detalle,
      data.table(
        iter = iteracion_actual,
        arbol = iarbol,
        testing = resultado$testing,
        testing_pos = resultado$testing_pos,
        envios = resultado$envios,
        aciertos = resultado$aciertos,
        ganancia = resultado$ganancia
      )
    )

    if (
      iarbol %% 8 == 0 ||
      iarbol == PARAM$num_trees_max
    ) {
      cat(
        "   árboles:",
        iarbol,
        "de",
        PARAM$num_trees_max,
        "\n"
      )
      flush.console()
    }
  }

  return(detalle)
}


## 4.03 Ejecución del Grid Search

La grilla tiene:

`3 feature_fraction × 4 maxdepth × 1 cp × 5 minsplit × 2 ratio_bucket = 120 combinaciones`

Cada combinación usa 32 árboles:

`120 × 32 = 3840 árboles`.


In [ ]:
tb_grid_search <- data.table()
tb_grid_search_detalle <- data.table()

qcombinaciones <- (
  length(PARAM$grid_feature_fraction) *
  length(PARAM$grid_maxdepth) *
  length(PARAM$grid_cp) *
  length(PARAM$grid_minsplit) *
  length(PARAM$grid_ratio_bucket)
)

iteracion_actual <- 0L

for (vmax_depth in PARAM$grid_maxdepth) {
  for (vcp in PARAM$grid_cp) {
    for (vmin_split in PARAM$grid_minsplit) {
      for (vratio_bucket in PARAM$grid_ratio_bucket) {
        for (vfeature_fraction in PARAM$grid_feature_fraction) {

          iteracion_actual <- iteracion_actual + 1L

          vmin_bucket <- as.integer(
            round(
              vmin_split * vratio_bucket
            )
          )

          cat(
            "\nIteración",
            iteracion_actual,
            "de",
            qcombinaciones,
            "| cp =", vcp,
            "| maxdepth =", vmax_depth,
            "| minsplit =", vmin_split,
            "| minbucket =", vmin_bucket,
            "| feature_fraction =", vfeature_fraction,
            "\n"
          )
          flush.console()

          param_rpart <- list(
            cp = vcp,
            maxdepth = vmax_depth,
            minsplit = vmin_split,
            minbucket = vmin_bucket
          )

          detalle_combo <- EvaluarCombinacion32(
            iteracion_actual = iteracion_actual,
            param_rpart = param_rpart,
            feature_fraction = vfeature_fraction
          )

          detalle_combo[
            ,
            `:=`(
              cp = vcp,
              maxdepth = vmax_depth,
              minsplit = vmin_split,
              minbucket = vmin_bucket,
              ratio_bucket = vratio_bucket,
              feature_fraction = vfeature_fraction
            )
          ]

          tb_grid_search_detalle <- rbind(
            tb_grid_search_detalle,
            detalle_combo,
            fill = TRUE
          )

          resumen_combo <- data.table(
            iter = iteracion_actual,
            cp = vcp,
            maxdepth = vmax_depth,
            minsplit = vmin_split,
            minbucket = vmin_bucket,
            ratio_bucket = vratio_bucket,
            feature_fraction = vfeature_fraction,
            num_trees = nrow(detalle_combo),
            ganancia_mean = mean(
              detalle_combo$ganancia
            ),
            ganancia_sd = sd(
              detalle_combo$ganancia
            ),
            ganancia_median = median(
              detalle_combo$ganancia
            ),
            ganancia_min = min(
              detalle_combo$ganancia
            ),
            ganancia_max = max(
              detalle_combo$ganancia
            ),
            envios_mean = mean(
              detalle_combo$envios
            ),
            aciertos_mean = mean(
              detalle_combo$aciertos
            )
          )

          tb_grid_search <- rbind(
            tb_grid_search,
            resumen_combo,
            fill = TRUE
          )

          cat(
            "   ganancia media:",
            round(
              resumen_combo$ganancia_mean
            ),
            "\n"
          )

          # guardado incremental
          fwrite(
            tb_grid_search,
            "gridsearch_resultados.csv"
          )

          fwrite(
            tb_grid_search_detalle,
            "gridsearch_detalle_32_arboles.csv"
          )

          gc(verbose = FALSE)
        }
      }
    }
  }
}


## 4.04 Ranking final

El criterio principal es `ganancia_mean`, es decir, el promedio de la ganancia normalizada de los 32 árboles de cada combinación.


In [22]:
ranking <- tb_grid_search[
  order(-ganancia_mean)
]

ranking[
  1:min(20, .N)
]


ERROR: Error: object 'tb_grid_search' not found


In [23]:
mejor <- ranking[1]

mejor


ERROR: Error: object 'ranking' not found


In [24]:
cat(
  "Combinaciones evaluadas:",
  nrow(tb_grid_search),
  "\n"
)

cat(
  "Árboles evaluados:",
  nrow(tb_grid_search_detalle),
  "\n"
)

# al finalizar deberían ser:
# 120 combinaciones
# 3840 árboles


ERROR: Error: object 'tb_grid_search' not found


In [ ]:
fwrite(
  ranking,
  "gridsearch_ranking.csv"
)

fwrite(
  tb_grid_search_detalle[
    order(iter, arbol)
  ],
  "gridsearch_detalle_32_arboles.csv"
)


## 4.05 Generar la predicción final para Kaggle

Una vez elegida la mejor combinación, se vuelve a entrenar con **todo 202107**.

Se fija una sola vez la misma `semilla_primigenia`, se generan 32 subconjuntos aleatorios de variables, se entrenan 32 árboles y se promedian sus probabilidades sobre `202109`.


In [9]:
GenerarSubmission32 <- function(
  param_rpart,
  feature_fraction
) {

  prob_acumulada <- rep(
    0,
    nrow(dfuture)
  )

  qty_campos <- max(
    1L,
    as.integer(
      length(campos_buenos) *
        feature_fraction
    )
  )

  # una sola semilla inicial
  set.seed(PARAM$semilla_primigenia)

  for (
    iarbol in seq_len(
      PARAM$num_trees_max
    )
  ) {

    campos_random <- sample(
      campos_buenos,
      qty_campos,
      replace = FALSE
    )

    formulita <- paste0(
      "clase_ternaria ~ ",
      paste(
        campos_random,
        collapse = " + "
      )
    )

    modelo <- rpart(
      formulita,
      data = dtrain_full,
      xval = 0,
      control = param_rpart
    )

    prediccion <- predict(
      modelo,
      dfuture,
      type = "prob"
    )

    prob_acumulada <- (
      prob_acumulada +
      prediccion[, "BAJA+2"]
    )

    if (
      iarbol %% 8 == 0 ||
      iarbol == PARAM$num_trees_max
    ) {
      cat(
        "   árbol final:",
        iarbol,
        "de",
        PARAM$num_trees_max,
        "\n"
      )
      flush.console()
    }
  }

  prob_promedio <- (
    prob_acumulada /
    PARAM$num_trees_max
  )

  return(
    data.table(
      numero_de_cliente =
        dfuture$numero_de_cliente,
      Predicted = as.numeric(
        prob_promedio >
          PARAM$prob_corte
      )
    )
  )
}


In [10]:
param_mejor <- list(
  cp = mejor$cp,
  maxdepth = mejor$maxdepth,
  minsplit = mejor$minsplit,
  minbucket = mejor$minbucket
)

submission_mejor <- GenerarSubmission32(
  param_rpart = param_mejor,
  feature_fraction =
    mejor$feature_fraction
)

archivo_final <- "KA_MEJOR_GRID_32_ARBOLES.csv"

fwrite(
  submission_mejor,
  archivo_final,
  sep = ","
)

cat(
  "\nArchivo final:",
  archivo_final,
  "\n"
)

cat(
  "Iteración ganadora:",
  mejor$iter,
  "\n"
)

cat(
  "cp:",
  mejor$cp,
  "\n"
)

cat(
  "maxdepth:",
  mejor$maxdepth,
  "\n"
)

cat(
  "minsplit:",
  mejor$minsplit,
  "\n"
)

cat(
  "minbucket:",
  mejor$minbucket,
  "\n"
)

cat(
  "feature_fraction:",
  mejor$feature_fraction,
  "\n"
)

cat(
  "ganancia_mean:",
  round(mejor$ganancia_mean),
  "\n"
)


ERROR: Error: object 'mejor' not found


In [11]:
format(
  Sys.time(),
  "%a %b %d %X %Y"
)


[1] "Mon Oct 05 17:28:02 2026"

In [25]:
# ============================================================
# RECONSTRUIR OBJETOS NECESARIOS PARA GENERAR SUBMISSIONS
# ============================================================

library(data.table)
library(rpart)

# parámetros necesarios para los 32 árboles finales
PARAM <- list()

PARAM$semilla_primigenia <- 149911
PARAM$num_trees_max <- 32L
PARAM$prob_corte <- 0.025

# cargar dataset
dataset <- fread(
  "/content/datasets/dataset_pequeno.csv"
)

# 202107 = período con clase conocida para entrenar
dtrain_full <- dataset[
  foto_mes == 202107
]

# 202109 = período a predecir para Kaggle
dfuture <- dataset[
  foto_mes == 202109
]

dfuture[
  ,
  clase_ternaria := NA
]

# campos que pueden entrar aleatoriamente en los árboles
campos_buenos <- setdiff(
  colnames(dtrain_full),
  c(
    "clase_ternaria",
    "fold"
  )
)

cat(
  "202107:",
  nrow(dtrain_full),
  "registros\n"
)

cat(
  "202109:",
  nrow(dfuture),
  "registros\n"
)

cat(
  "Campos candidatos:",
  length(campos_buenos),
  "\n"
)

202107: 164479 registros
202109: 165076 registros
Campos candidatos: 154 


In [27]:
GenerarSubmission32 <- function(
  param_rpart,
  feature_fraction
) {

  prob_acumulada <- rep(
    0,
    nrow(dfuture)
  )

  qty_campos <- max(
    1L,
    as.integer(
      length(campos_buenos) *
        feature_fraction
    )
  )

  # misma semilla inicial que usamos en el grid
  set.seed(PARAM$semilla_primigenia)

  for (
    iarbol in seq_len(
      PARAM$num_trees_max
    )
  ) {

    campos_random <- sample(
      campos_buenos,
      qty_campos,
      replace = FALSE
    )

    formulita <- paste0(
      "clase_ternaria ~ ",
      paste(
        campos_random,
        collapse = " + "
      )
    )

    modelo <- rpart(
      formulita,
      data = dtrain_full,
      xval = 0,
      control = param_rpart
    )

    prediccion <- predict(
      modelo,
      dfuture,
      type = "prob"
    )

    prob_acumulada <- (
      prob_acumulada +
      prediccion[, "BAJA+2"]
    )

    if (
      iarbol %% 8 == 0 ||
      iarbol == PARAM$num_trees_max
    ) {

      cat(
        "   árbol:",
        iarbol,
        "de",
        PARAM$num_trees_max,
        "\n"
      )

      flush.console()
    }
  }

  prob_promedio <- (
    prob_acumulada /
      PARAM$num_trees_max
  )

  return(
    data.table(
      numero_de_cliente =
        dfuture$numero_de_cliente,

      Predicted = as.numeric(
        prob_promedio >
          PARAM$prob_corte
      )
    )
  )
}

In [ ]:
# ============================================================
# CARGAR RANKING + GENERAR + SUBIR TOP 10 A KAGGLE
# ============================================================

library(data.table)

# ------------------------------------------------------------
# 1. Cargar ranking ya guardado
# ------------------------------------------------------------

ranking <- fread(
  "/content/buckets/b1/exp/exp4021_gridsearch_semilla_unica/gridsearch_ranking.csv"
)

# por seguridad, vuelvo a ordenar
ranking <- ranking[
  order(-ganancia_mean)
]

# tomo las 10 mejores combinaciones
top10 <- ranking[
  1:min(10, .N)
]

cat("\nTOP 10 A SUBIR:\n")

print(
  top10[
    ,
    .(
      iter,
      cp,
      maxdepth,
      minsplit,
      minbucket,
      feature_fraction,
      ganancia_mean
    )
  ]
)

# ------------------------------------------------------------
# 2. Competencia Kaggle
# ------------------------------------------------------------

competencia <- "labo-1-virtual-inicial"

# tabla para registrar resultados de las subidas
tb_subidas_top10 <- data.table()

# ------------------------------------------------------------
# 3. Generar y subir cada una de las 10 mejores
# ------------------------------------------------------------

for (i in seq_len(nrow(top10))) {

  config <- top10[i]

  cat(
    "\n============================================\n",
    "TOP", i, "de", nrow(top10), "\n",
    "iter =", config$iter, "\n",
    "cp =", config$cp, "\n",
    "maxdepth =", config$maxdepth, "\n",
    "minsplit =", config$minsplit, "\n",
    "minbucket =", config$minbucket, "\n",
    "feature_fraction =", config$feature_fraction, "\n",
    "ganancia_mean =", round(config$ganancia_mean), "\n",
    "============================================\n"
  )

  # hiperparámetros de rpart
  param_top <- list(
    cp = config$cp,
    maxdepth = config$maxdepth,
    minsplit = config$minsplit,
    minbucket = config$minbucket
  )

  # ----------------------------------------------------------
  # Genero la predicción final:
  # 32 árboles sobre TODO 202107
  # ----------------------------------------------------------

  submission_top <- GenerarSubmission32(
    param_rpart = param_top,
    feature_fraction = config$feature_fraction
  )

  # ----------------------------------------------------------
  # Nombre del archivo
  # ----------------------------------------------------------

  archivo_kaggle <- paste0(
    "KA_TOP",
    sprintf("%02d", i),
    "_iter",
    sprintf("%03d", config$iter),
    ".csv"
  )

  fwrite(
    submission_top,
    file = archivo_kaggle,
    sep = ","
  )

  # ----------------------------------------------------------
  # Mensaje Kaggle
  # ----------------------------------------------------------

  mensaje <- paste0(
    "-m 'TOP", i,
    " iter=", config$iter,
    " cp=", config$cp,
    " maxdepth=", config$maxdepth,
    " minsplit=", config$minsplit,
    " minbucket=", config$minbucket,
    " feature_fraction=", config$feature_fraction,
    " ganancia_mean=", round(config$ganancia_mean),
    "'"
  )

  # ----------------------------------------------------------
  # Comando Kaggle
  # ----------------------------------------------------------

  comando <- "kaggle competitions submit"

  comp <- paste(
    "-c",
    competencia
  )

  arch <- paste(
    "-f",
    archivo_kaggle
  )

  linea <- paste(
    comando,
    comp,
    arch,
    mensaje
  )

  cat(
    "\nSubiendo:",
    archivo_kaggle,
    "\n"
  )

  # ----------------------------------------------------------
  # Subida
  # ----------------------------------------------------------

  salida <- tryCatch(
    {
      system(
        linea,
        intern = TRUE
      )
    },
    error = function(e) {
      paste(
        "ERROR:",
        e$message
      )
    }
  )

  cat(
    paste(
      salida,
      collapse = "\n"
    ),
    "\n"
  )

  # ----------------------------------------------------------
  # Registro
  # ----------------------------------------------------------

  tb_subidas_top10 <- rbind(
    tb_subidas_top10,
    data.table(
      ranking = i,
      iter = config$iter,
      cp = config$cp,
      maxdepth = config$maxdepth,
      minsplit = config$minsplit,
      minbucket = config$minbucket,
      feature_fraction = config$feature_fraction,
      ganancia_mean = config$ganancia_mean,
      archivo = archivo_kaggle
    ),
    fill = TRUE
  )

  # pausa corta entre submissions
  Sys.sleep(3)
}

# ------------------------------------------------------------
# 4. Guardar registro de lo subido
# ------------------------------------------------------------

fwrite(
  tb_subidas_top10,
  "subidas_top10_kaggle.csv"
)

cat(
  "\n============================================\n",
  "FINALIZADO\n",
  "Se generaron/subieron",
  nrow(tb_subidas_top10),
  "submissions.\n",
  "============================================\n"
)

tb_subidas_top10


TOP 10 A SUBIR:
     iter    cp maxdepth minsplit minbucket feature_fraction ganancia_mean
    <int> <int>    <int>    <int>     <int>            <num>         <num>
 1:    63    -1        8      200        67              0.9     491393229
 2:    47    -1        7      600       240              0.7     491328125
 3:    66    -1        8      200        80              0.9     491223958
 4:     2    -1        6      200        67              0.7     489804688
 5:    50    -1        7      800       267              0.7     489333333
 6:   105    -1        9      600       200              0.9     489221354
 7:   107    -1        9      600       240              0.7     489091146
 8:    77    -1        8      600       240              0.7     488950521
 9:     8    -1        6      400       133              0.7     488385417
10:    75    -1        8      600       200              0.9     487664062

 TOP 1 de 10 
 iter = 63 
 cp = -1 
 maxdepth = 8 
 minsplit = 200 
 minbucket = 6